In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.downloads.density_calculator_optimized import quick_run, validate_inputs, show_result_stats

In [ ]:
SENTINEL_IMG = "/content/drive/MyDrive/Fine-Tuning-Workflow/Downloads/S2/s2_2025_partidos_ambas.tif"
BUILDINGS_GEOJSON = "/content/drive/MyDrive/ML_Inference_Data/morphometric_data/AMBA_buildings_complete.geojson"
OUTPUT_TIFF = "/content/drive/MyDrive/Fine-Tuning-Workflow/Downloads/S2/s2_2025_building_density.tif"

In [ ]:

def run():
    """Ejecuta el cálculo de densidad de forma simple"""
    success = quick_run(
        sentinel_img=SENTINEL_IMG,
        buildings_geojson=BUILDINGS_GEOJSON,
        output_tiff=OUTPUT_TIFF,
        tile_size=2000,    # Reducir si hay problemas de memoria
        batch_size=2000    # Reducir si hay problemas de memoria
    )

    if success:
        print(f"✅ Completado! Resultado en: {OUTPUT_TIFF}")
    else:
        print("❌ Error en el procesamiento")

    return success

In [ ]:
validate_inputs(SENTINEL_IMG, BUILDINGS_GEOJSON, OUTPUT_TIFF)

True

In [ ]:
run()


🚀 INICIANDO CÁLCULO DE DENSIDAD OPTIMIZADO


Procesando tiles:  52%|█████▏    | 47/90 [4:55:54<5:12:11, 435.61s/it, Features=4,707,519, Tile=47/90]

# Densidad Multiescala

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.downloads.multiscale_density_calculator import (
    quick_run_multiscale,
    create_multiscale_density_stack,
    MultiScaleDensityCalculator
)

In [2]:
SENTINEL_IMG = '/content/drive/MyDrive/ML_Inference_Data/s2_composite_unormalized.tif'
BUILDINGS_GEOJSON =  "/content/drive/MyDrive/ML_Inference_Data/morphometric_data/AMBA_buildings_complete.geojson"
OUTPUT_DIR = '/content/drive/MyDrive/Fine-Tuning-Workflow/Downloads/S2/density_multiscale_AMBA/'


In [3]:
stack, band_names = quick_run_multiscale(
    sentinel_img=SENTINEL_IMG,
    buildings_geojson=BUILDINGS_GEOJSON,
    output_dir=OUTPUT_DIR,
    resolutions=[10, 50, 100]
)


🚀 INICIANDO CÁLCULO MULTI-ESCALA CON GDAL


   Copiando bandas: 100%|██████████| 7/7 [03:18<00:00, 28.31s/it]


🎉 PROCESAMIENTO COMPLETADO
📦 Stack shape: (11237, 11563, 7)
📁 Archivos guardados en: /content/drive/MyDrive/Fine-Tuning-Workflow/Downloads/S2/density_multiscale_AMBA/

🎯 Archivo principal para tu modelo:
   /content/drive/MyDrive/Fine-Tuning-Workflow/Downloads/S2/density_multiscale_AMBA//density_stack_multiband.tif


In [4]:
import numpy as np

# ¿Cuánto ocupa tu density_array?
print(f"Shape de density_array: {calculator.density_array.shape}")
print(f"Tamaño en RAM: {calculator.density_array.nbytes / 1024**3:.2f} GB")

# ¿Cuánto ocuparía el zoom?
factor = 3  # para density_30m
h, w = calculator.density_array.shape
down_h, down_w = h // factor, w // factor
print(f"\nArray downsampled: ({down_h}, {down_w})")
print(f"Tamaño: {(down_h * down_w * 4) / 1024**3:.2f} GB")
print(f"\nArray upsampled: ({h}, {w})")
print(f"Tamaño: {(h * w * 4) / 1024**3:.2f} GB")

NameError: name 'calculator' is not defined

# New Morphometrics

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Setup complete!")

# =============================================================================
# CELL 2: CONFIGURATION (Check paths are correct)
# =============================================================================

# Your file paths
BUILDINGS_GEOJSON = "/content/drive/MyDrive/ML_Inference_Data/morphometric_data/95b_buildings.geojson"
REFERENCE_RASTER = "/content/drive/MyDrive/Fine-Tuning-Workflow/Downloads/S2/s2_2025_partidos_ambas.tif"
OUTPUT_DIR = "/content/drive/MyDrive/ML_Inference_Data/new_morphometric_data_partidos_amba/"

# Settings
NEIGHBORHOOD_RADIUS = 256  # meters

print("Configuration:")
print(f"   Buildings: {BUILDINGS_GEOJSON}")
print(f"   Reference: {REFERENCE_RASTER}")
print(f"   Output: {OUTPUT_DIR}")
print(f"   Neighborhood: {NEIGHBORHOOD_RADIUS}m")

from src.downloads.morphometric_calculator import MorphometricCalculator, show_morphometric_stats

In [ ]:
# =============================================================================
# CELL 3: RUN MORPHOMETRICS (This does all the work)
# =============================================================================

print("🚀 Starting morphometric calculations...")
print("This will take a while for large datasets - be patient!")
print("="*60)

from morphometric_calculator import MorphometricCalculator, show_morphometric_stats

# Initialize
calculator = MorphometricCalculator(
    buildings_geojson_path=BUILDINGS_GEOJSON,
    reference_raster_path=REFERENCE_RASTER,
    output_dir=OUTPUT_DIR,
    neighborhood_radius=NEIGHBORHOOD_RADIUS
)

# Validate setup
if not calculator.validate_setup():
    print("❌ Setup validation failed - check your file paths!")
else:
    print("✅ Validation passed - starting calculations...\n")

    # Calculate all 3 morphometrics
    results = calculator.calculate_all_morphometrics()

    # Show results
    print("\n" + "="*60)
    print("🎉 PROCESSING COMPLETE!")
    print("="*60)

    for metric_name, output_path in results.items():
        if output_path:
            print(f"✅ {metric_name}")
            show_morphometric_stats(output_path)
            print()
        else:
            print(f"❌ {metric_name} - FAILED")

    success_count = len([r for r in results.values() if r is not None])
    print(f"\n📊 Success: {success_count}/3 morphometrics")

    if success_count == 3:
        print("🎉 All morphometrics ready for CNN training!")
        print(f"📁 Files saved to: {OUTPUT_DIR}")
        print("\n📋 Output files:")
        print("   • orientation_variance.tif")
        print("   • spacing_irregularity.tif")
        print("   • size_heterogeneity.tif")
    else:
        print("⚠️  Some morphometrics failed - check logs above")

# =============================================================================
# DONE! Your 3 morphometric rasters are ready.
# Use them as separate channels in your multi-branch CNN alongside Sentinel-2
# =============================================================================

🚀 Starting morphometric calculations...
This will take a while for large datasets - be patient!
✅ Validation passed - starting calculations...



Tiles (orientation):   0%|          | 0/342 [00:00<?, ?it/s]

  0%|          | 0/159 [00:00<?, ?it/s]

Tiles (orientation):   0%|          | 1/342 [05:51<33:20:14, 351.95s/it]

  0%|          | 0/76 [00:00<?, ?it/s]

...

Tiles (orientation):  64%|██████▍   | 220/342 [22:16:10<12:54:51, 381.08s/it]

  0%|          | 0/175033 [00:00<?, ?it/s]

Tiles (orientation):  65%|██████▍   | 221/342 [22:22:39<12:53:39, 383.63s/it]

  0%|          | 0/190168 [00:00<?, ?it/s]

Tiles (orientation):  65%|██████▍   | 222/342 [22:29:19<12:56:42, 388.35s/it]